In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from datetime import datetime
import hashlib


MATRIX_PRESET = "cartesian"
USE_USPLAT = False
DROPOUT_DEFAULT = "no_dropout"


FINAL_COLUMNS = [
    "status",
    "run_hash",
    "scene_name",
    "variant_name",
    "variant_index",
    "matrix_preset",
    "isotropy",
    "appearance",
    "sorting",
    "pruning",
    "dropout",
    "ess",
    "use_usplat",
    "config_path",
    "generated_config_path",
    "model_path",
    "checkpoint_index",
    "checkpoint_filename",
    "checkpoint_path",
    "checkpoint_name_iteration",
    "eval_requested_split",
    "eval_split_used",
    "eval_checkpoint_iteration",
    "psnr",
    "ssim",
    "lpips",
    "render_fps",
    "peak_eval_vram_mb",
    "final_gaussian_count",
    "checkpoint_size_bytes",
    "checkpoint_size_mb",
    "training_wall_clock_sec_at_checkpoint",
    "metrics_jsonl_path",
    "metrics_csv_path",
    "created_at",
]


def stable_hash(text: str, length: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:length]


def parse_variant_dirname(dirname: str) -> dict:
    """
    Supports:

    5-part:
        isotropic--rgb--sort--no_pruning--no_ess

    6-part:
        isotropic--rgb--sort--no_pruning--no_dropout--no_ess

    7-part:
        anisotropic--rgb--sort--interleaved_prune_densify--ess--dropout--no_usplat
    """
    parts = dirname.split("--")

    use_usplat = USE_USPLAT

    if len(parts) == 5:
        isotropy, appearance, sorting, pruning, ess = parts
        dropout = DROPOUT_DEFAULT

    elif len(parts) == 6:
        isotropy, appearance, sorting, pruning, dropout, ess = parts

    elif len(parts) == 7:
        isotropy, appearance, sorting, pruning, ess, dropout, usplat = parts
        use_usplat = usplat != "no_usplat"

    else:
        raise ValueError(f"Unexpected variant folder name: {dirname}")

    variant_parts = [
        isotropy,
        appearance,
        sorting,
        pruning,
        dropout,
        ess,
    ]

    return {
        "isotropy": isotropy,
        "appearance": appearance,
        "sorting": sorting,
        "pruning": pruning,
        "dropout": dropout,
        "ess": ess,
        "use_usplat": use_usplat,
        "variant_name": "__".join(variant_parts),
    }

def value_or(record: dict, key: str, fallback):
    value = record.get(key, pd.NA)
    return fallback if pd.isna(value) or value == "" else value


def read_dataset_results(dataset_dir: Path, write_csv: bool = True) -> pd.DataFrame:
    dataset_dir = Path(dataset_dir)
    scene_name = dataset_dir.name
    config_path = f"configs/dnerf_ablation/{scene_name}.yaml"
    output_csv = dataset_dir.parent / f"{scene_name}_checkpoint_eval_metrics_all.csv"

    rows = []
    created_at = datetime.now().replace(microsecond=0).isoformat()

    variant_dirs = sorted(p for p in dataset_dir.iterdir() if p.is_dir())

    for variant_index, variant_dir in enumerate(variant_dirs):
        metrics_csv = variant_dir / "checkpoint_eval_metrics.csv"

        if not metrics_csv.exists():
            print(f"Skipping, no metrics file: {metrics_csv}")
            continue

        parsed = parse_variant_dirname(variant_dir.name)

        df = pd.read_csv(metrics_csv, skipinitialspace=True)
        df.columns = df.columns.str.strip()

        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
                df[col] = df[col].astype("string").str.strip()

        for _, row in df.iterrows():
            record = row.to_dict()

            record["run_hash"] = value_or(
                record,
                "run_hash",
                stable_hash(str(variant_dir.resolve())),
            )

            record["scene_name"] = value_or(
                record,
                "scene_name",
                scene_name,
            )

            record["variant_name"] = value_or(
                record,
                "variant_name",
                parsed["variant_name"],
            )

            record["variant_index"] = variant_index
            record["matrix_preset"] = MATRIX_PRESET

            for key in ["isotropy", "appearance", "sorting", "pruning", "dropout", "ess"]:
                record[key] = value_or(record, key, parsed[key])

            record["use_usplat"] = USE_USPLAT
            record["config_path"] = config_path

            record["model_path"] = value_or(
                record,
                "model_path",
                str(variant_dir.resolve()),
            )

            record["metrics_csv_path"] = str(metrics_csv.resolve())
            record["metrics_jsonl_path"] = str(
                (variant_dir / "checkpoint_eval_metrics.jsonl").resolve()
            )

            record["training_wall_clock_sec_at_checkpoint"] = value_or(
                record,
                "training_wall_clock_sec_at_checkpoint",
                pd.NA,
            )

            # chkpnt_best.pth may have no number in filename, so use loaded eval iteration.
            record["checkpoint_name_iteration"] = value_or(
                record,
                "checkpoint_name_iteration",
                record.get("eval_checkpoint_iteration", pd.NA),
            )

            record["created_at"] = created_at
            rows.append(record)

    combined = pd.DataFrame(rows)

    for col in FINAL_COLUMNS:
        if col not in combined.columns:
            combined[col] = pd.NA

    byte_cols = [
        c for c in combined.columns
        if c.endswith("_bytes") or "bytes" in c.lower()
    ]

    for col in byte_cols:
        mb_col = col.replace("_bytes", "_mb")
        combined[mb_col] = pd.to_numeric(combined[col], errors="coerce") / (1024 ** 2)

    for col in FINAL_COLUMNS:
        if col not in combined.columns:
            combined[col] = pd.NA

    final_df = combined[FINAL_COLUMNS].copy()

    if write_csv:
        final_df.to_csv(output_csv, index=False)
        print(f"Wrote {len(final_df)} rows to {output_csv}")

    return final_df


def read_all_dataset_results(output_root: Path = Path("output")) -> dict[str, pd.DataFrame]:
    results = {}

    for ablation_dir in sorted(output_root.glob("ablations_*")):
        if not ablation_dir.is_dir():
            continue

        dataset_dirs = sorted(
            p for p in ablation_dir.iterdir()
            if p.is_dir() and p.name != "generated_configs"
        )

        for dataset_dir in dataset_dirs:
            key = f"{ablation_dir.name}/{dataset_dir.name}"
            results[key] = read_dataset_results(dataset_dir)

    return results

dataset_results = read_all_dataset_results()
dataset_results["ablations_usplat_sort_dropout_off/trex"]["use_usplat"] = True
dataset_results["ablations_no_usplat_dropout_on/trex"]["dropout"] = "yes_dropout"

def infer_use_usplat(row):
    tokens = []

    if pd.notna(row.get("variant_name")):
        tokens += str(row["variant_name"]).split("__")

    if pd.notna(row.get("model_path")):
        tokens += Path(str(row["model_path"])).name.split("--")

    if pd.notna(row.get("generated_config_path")):
        tokens += Path(str(row["generated_config_path"])).stem.split("__")

    return "use_usplat" in tokens

dataset_results["ablations_bouncing_balls_usplat_vs_no_usplat_7k/bouncingballs"]["use_usplat"] = dataset_results["ablations_bouncing_balls_usplat_vs_no_usplat_7k/bouncingballs"].apply(infer_use_usplat, axis=1)


def merge_dataset_results(
    dataset_results: dict[str, pd.DataFrame],
    drop_rules: list[tuple[str, str, object]] | None = None,
) -> pd.DataFrame:
    drop_rules = drop_rules or []
    rules_by_key = {}

    for key, col, value in drop_rules:
        rules_by_key.setdefault(key, []).append((col, value))

    merged = []

    for key, df in dataset_results.items():
        df = df.copy()
        original_len = len(df)

        drop_mask = pd.Series(False, index=df.index)

        for col, value in rules_by_key.get(key, []):
            if col not in df.columns:
                print(f"{key}: skip missing drop column: {col}")
                continue

            drop_mask |= df[col].astype(str).str.lower().eq(str(value).lower())

        dropped = int(drop_mask.sum())
        kept = df.loc[~drop_mask].copy()

        frac = dropped / original_len if original_len else 0.0
        print(f"{key}: dropped {dropped}/{original_len} rows ({frac:.1%})")

        kept.insert(0, "dataset_key", key)
        merged.append(kept)

    final_df = pd.concat(merged, ignore_index=True) if merged else pd.DataFrame()
    return final_df


final_df = merge_dataset_results(
    dataset_results,
    drop_rules=[
        ("ablations_no_usplat_dropout_off/trex", "sorting", "sort_free"),
    ],
)

Wrote 704 rows to output/ablations_bouncing_balls_no_usplat/bouncingballs_checkpoint_eval_metrics_all.csv
Skipping, no metrics file: output/ablations_bouncing_balls_usplat_5k/bouncingballs/anisotropic--rgb--sort--no_pruning--no_ess--no_dropout/checkpoint_eval_metrics.csv
Wrote 0 rows to output/ablations_bouncing_balls_usplat_5k/bouncingballs_checkpoint_eval_metrics_all.csv
Wrote 352 rows to output/ablations_bouncing_balls_usplat_vs_no_usplat_7k/bouncingballs_checkpoint_eval_metrics_all.csv
Wrote 672 rows to output/ablations_fixed_longer/trex_checkpoint_eval_metrics_all.csv
Wrote 672 rows to output/ablations_fixed_longer_sortfree/trex_checkpoint_eval_metrics_all.csv
Wrote 352 rows to output/ablations_no_usplat_dropout_off/trex_checkpoint_eval_metrics_all.csv
Wrote 352 rows to output/ablations_no_usplat_dropout_on/trex_checkpoint_eval_metrics_all.csv
Skipping, no metrics file: output/ablations_usplat_sort_dropout_off/trex/anisotropic--rgb--sort--no_pruning--ess/checkpoint_eval_metrics.cs

In [2]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import hashlib

import pandas as pd
import numpy as np


MATRIX_PRESET = "cartesian"
USE_USPLAT_DEFAULT = False
DROPOUT_DEFAULT = "no_dropout"


FINAL_COLUMNS = [
    "status",
    "run_hash",
    "scene_name",
    "variant_name",
    "variant_index",
    "matrix_preset",
    "isotropy",
    "appearance",
    "sorting",
    "pruning",
    "dropout",
    "ess",
    "use_usplat",
    "config_path",
    "generated_config_path",
    "model_path",
    "checkpoint_index",
    "checkpoint_filename",
    "checkpoint_path",
    "checkpoint_name_iteration",
    "eval_requested_split",
    "eval_split_used",
    "eval_checkpoint_iteration",
    "psnr",
    "ssim",
    "lpips",
    "render_fps",
    "peak_eval_vram_mb",
    "final_gaussian_count",
    "checkpoint_size_bytes",
    "checkpoint_size_mb",
    "training_wall_clock_sec_at_checkpoint",
    "metrics_jsonl_path",
    "metrics_csv_path",
    "created_at",
]


DATASET_OVERRIDES = {
    "ablations_usplat_sort_dropout_off/trex": {
        "use_usplat": True,
    },
    "ablations_no_usplat_dropout_on/trex": {
        "dropout": "yes_dropout",
    },
}

INFER_USPLAT_KEYS = {
    "ablations_bouncing_balls_usplat_vs_no_usplat_7k/bouncingballs",
}


DROP_RULES = [
    ("ablations_no_usplat_dropout_off/trex", "sorting", "sort_free"),
]


def stable_hash(text: str, length: int = 16) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:length]


def is_missing(value) -> bool:
    if value is None:
        return True

    try:
        if pd.isna(value):
            return True
    except (TypeError, ValueError):
        pass

    return isinstance(value, str) and value.strip() == ""

def value_or(record: dict, key: str, fallback):
    value = record.get(key, pd.NA)
    return fallback if is_missing(value) else value


def parse_variant_dirname(dirname: str) -> dict:
    """
    Supports:

    5-part:
        isotropic--rgb--sort--no_pruning--no_ess

    6-part:
        isotropic--rgb--sort--no_pruning--no_dropout--no_ess

    7-part:
        anisotropic--rgb--sort--interleaved_prune_densify--ess--dropout--no_usplat
    """
    parts = dirname.split("--")
    use_usplat = USE_USPLAT_DEFAULT

    if len(parts) == 5:
        isotropy, appearance, sorting, pruning, ess = parts
        dropout = DROPOUT_DEFAULT

    elif len(parts) == 6:
        isotropy, appearance, sorting, pruning, dropout, ess = parts

    elif len(parts) == 7:
        isotropy, appearance, sorting, pruning, ess, dropout, usplat = parts
        use_usplat = usplat != "no_usplat"

    else:
        raise ValueError(f"Unexpected variant folder name: {dirname}")

    variant_name = "__".join(
        [isotropy, appearance, sorting, pruning, dropout, ess]
    )

    return {
        "isotropy": isotropy,
        "appearance": appearance,
        "sorting": sorting,
        "pruning": pruning,
        "dropout": dropout,
        "ess": ess,
        "use_usplat": use_usplat,
        "variant_name": variant_name,
    }


def clean_string_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.str.strip()

    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].astype("string").str.strip()

    return df


def ensure_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        if col not in df.columns:
            df[col] = pd.NA
    return df


def add_mb_columns(df: pd.DataFrame) -> pd.DataFrame:
    byte_cols = [
        col for col in df.columns
        if col.endswith("_bytes") or "bytes" in col.lower()
    ]

    for byte_col in byte_cols:
        mb_col = byte_col.replace("_bytes", "_mb")
        mb_values = pd.to_numeric(df[byte_col], errors="coerce") / (1024 ** 2)

        if mb_col in df.columns:
            df[mb_col] = df[mb_col].where(pd.notna(df[mb_col]), mb_values)
        else:
            df[mb_col] = mb_values

    return df


def build_record(
    row: pd.Series,
    *,
    variant_dir: Path,
    variant_index: int,
    parsed: dict,
    scene_name: str,
    config_path: str,
    created_at: str,
) -> dict:
    record = row.to_dict()

    defaults = {
        "run_hash": stable_hash(str(variant_dir.resolve())),
        "scene_name": scene_name,
        "variant_name": parsed["variant_name"],
        "matrix_preset": MATRIX_PRESET,
        "model_path": str(variant_dir.resolve()),
        "metrics_csv_path": str((variant_dir / "checkpoint_eval_metrics.csv").resolve()),
        "metrics_jsonl_path": str((variant_dir / "checkpoint_eval_metrics.jsonl").resolve()),
        "training_wall_clock_sec_at_checkpoint": pd.NA,
    }

    for key in ["isotropy", "appearance", "sorting", "pruning", "dropout", "ess", "use_usplat"]:
        defaults[key] = parsed[key]

    for key, fallback in defaults.items():
        record[key] = value_or(record, key, fallback)

    record["variant_index"] = variant_index
    record["config_path"] = config_path
    record["created_at"] = created_at

    # chkpnt_best.pth may have no number in filename, so use loaded eval iteration.
    record["checkpoint_name_iteration"] = value_or(
        record,
        "checkpoint_name_iteration",
        record.get("eval_checkpoint_iteration", pd.NA),
    )

    return record


def read_dataset_results(dataset_dir: Path, write_csv: bool = True) -> pd.DataFrame:
    dataset_dir = Path(dataset_dir)
    scene_name = dataset_dir.name

    config_path = f"configs/dnerf_ablation/{scene_name}.yaml"
    output_csv = dataset_dir.parent / f"{scene_name}_checkpoint_eval_metrics_all.csv"

    rows = []
    created_at = datetime.now().replace(microsecond=0).isoformat()

    variant_dirs = sorted(p for p in dataset_dir.iterdir() if p.is_dir())

    for variant_index, variant_dir in enumerate(variant_dirs):
        metrics_csv = variant_dir / "checkpoint_eval_metrics.csv"

        if not metrics_csv.exists():
            print(f"Skipping, no metrics file: {metrics_csv}")
            continue

        parsed = parse_variant_dirname(variant_dir.name)
        df = clean_string_columns(pd.read_csv(metrics_csv, skipinitialspace=True))

        for _, row in df.iterrows():
            rows.append(
                build_record(
                    row,
                    variant_dir=variant_dir,
                    variant_index=variant_index,
                    parsed=parsed,
                    scene_name=scene_name,
                    config_path=config_path,
                    created_at=created_at,
                )
            )

    combined = pd.DataFrame(rows)
    combined = ensure_columns(combined, FINAL_COLUMNS)
    combined = add_mb_columns(combined)
    combined = ensure_columns(combined, FINAL_COLUMNS)

    final_df = combined[FINAL_COLUMNS].copy()

    if write_csv:
        final_df.to_csv(output_csv, index=False)
        print(f"Wrote {len(final_df)} rows to {output_csv}")

    return final_df


def read_all_dataset_results(
    output_root: Path = Path("output"),
    write_csv: bool = True,
) -> dict[str, pd.DataFrame]:
    results = {}

    for ablation_dir in sorted(output_root.glob("ablations_*")):
        if not ablation_dir.is_dir():
            continue

        dataset_dirs = sorted(
            p for p in ablation_dir.iterdir()
            if p.is_dir() and p.name != "generated_configs"
        )

        for dataset_dir in dataset_dirs:
            key = f"{ablation_dir.name}/{dataset_dir.name}"
            results[key] = read_dataset_results(dataset_dir, write_csv=write_csv)

    return results


def infer_use_usplat(row: pd.Series) -> bool:
    tokens = []

    for col in ["variant_name", "model_path", "generated_config_path"]:
        value = row.get(col)

        if pd.isna(value):
            continue

        value = str(value)

        if col == "model_path":
            tokens.extend(Path(value).name.split("--"))
        elif col == "generated_config_path":
            tokens.extend(Path(value).stem.split("__"))
        else:
            tokens.extend(value.split("__"))

    return "use_usplat" in tokens


def apply_dataset_overrides(dataset_results: dict[str, pd.DataFrame]) -> None:
    for key, overrides in DATASET_OVERRIDES.items():
        if key not in dataset_results:
            print(f"Skipping missing override dataset: {key}")
            continue

        for col, value in overrides.items():
            dataset_results[key][col] = value

    for key in INFER_USPLAT_KEYS:
        if key not in dataset_results:
            print(f"Skipping missing inferred usplat dataset: {key}")
            continue

        dataset_results[key]["use_usplat"] = dataset_results[key].apply(
            infer_use_usplat,
            axis=1,
        )


def merge_dataset_results(
    dataset_results: dict[str, pd.DataFrame],
    drop_rules: list[tuple[str, str, object]] | None = None,
) -> pd.DataFrame:
    rules_by_key: dict[str, list[tuple[str, object]]] = {}

    for key, col, value in drop_rules or []:
        rules_by_key.setdefault(key, []).append((col, value))

    merged = []

    for key, df in dataset_results.items():
        df = df.copy()
        original_len = len(df)

        drop_mask = pd.Series(False, index=df.index)

        for col, value in rules_by_key.get(key, []):
            if col not in df.columns:
                print(f"{key}: skip missing drop column: {col}")
                continue

            drop_mask |= df[col].astype(str).str.lower().eq(str(value).lower())

        dropped = int(drop_mask.sum())
        kept = df.loc[~drop_mask].copy()

        frac = dropped / original_len if original_len else 0.0
        print(f"{key}: dropped {dropped}/{original_len} rows ({frac:.1%})")

        kept.insert(0, "dataset_key", key)
        merged.append(kept)

    return pd.concat(merged, ignore_index=True) if merged else pd.DataFrame()


ABLATION_BOOL_COLS = [
    "isotropy",
    "use_usplat",
    "appearance",
    "sorting",
    "pruning",
    "dropout",
    "ess",
]

CHECKPOINT_COLS = [
    "checkpoint_index",
    "checkpoint_name_iteration",
    "eval_checkpoint_iteration",
]

MODEL_SPEC_COLS = [
    "run_hash",
    "dataset_key",
    "scene_name",
    "variant_name",
    "variant_index",
    "matrix_preset",
    *ABLATION_BOOL_COLS,
    *CHECKPOINT_COLS,
]

ABLATION_COLS = [
    "scene_name",
    *ABLATION_BOOL_COLS,
    "eval_checkpoint_iteration",
]

RADAR_SPECS = {
    "+PSNR": {"col": "psnr", "higher_is_better": True},
    "+SSIM": {"col": "ssim", "higher_is_better": True},
    "-LPIPS": {"col": "lpips", "higher_is_better": False},
    "+FPS": {"col": "render_fps", "higher_is_better": True},
    "-VRAM": {"col": "peak_eval_vram_mb", "higher_is_better": False},
    "-Size": {"col": "checkpoint_size_mb", "higher_is_better": False},
    "-Time": {"col": "training_wall_clock_sec_at_checkpoint", "higher_is_better": False},
    "-Gaussians": {"col": "final_gaussian_count", "higher_is_better": False},
}

METRIC_COLS = [spec["col"] for spec in RADAR_SPECS.values()]

BOOLEAN_LABEL_PAIRS = {
    "isotropy": ("isotropy", "anisotropy"),
    "appearance": ("appearance", "no appearance"),
    "sorting": ("sorting", "no sorting"),
    "pruning": ("pruning", "no pruning"),
    "dropout": ("dropout", "no dropout"),
    "ess": ("ess", "no ess"),
    "use_usplat": ("usplat", "no usplat"),
    "uncertain": ("uncertain", "no uncertain"),
}

BOOLEAN_LABEL_MAP = {
    col: {
        True: true_label,
        False: false_label,
    }
    for col, (true_label, false_label) in BOOLEAN_LABEL_PAIRS.items()
}


dataset_results = read_all_dataset_results()
apply_dataset_overrides(dataset_results)

final_df = merge_dataset_results(
    dataset_results,
    drop_rules=DROP_RULES,
)

keep_cols = [c for c in MODEL_SPEC_COLS + METRIC_COLS if c in final_df.columns]
print("Keep:", keep_cols)

df_clean = final_df[keep_cols].copy()

print("\nShape: ", df_clean.shape)

Wrote 704 rows to output/ablations_bouncing_balls_no_usplat/bouncingballs_checkpoint_eval_metrics_all.csv
Skipping, no metrics file: output/ablations_bouncing_balls_usplat_5k/bouncingballs/anisotropic--rgb--sort--no_pruning--no_ess--no_dropout/checkpoint_eval_metrics.csv
Wrote 0 rows to output/ablations_bouncing_balls_usplat_5k/bouncingballs_checkpoint_eval_metrics_all.csv
Wrote 352 rows to output/ablations_bouncing_balls_usplat_vs_no_usplat_7k/bouncingballs_checkpoint_eval_metrics_all.csv
Wrote 672 rows to output/ablations_fixed_longer/trex_checkpoint_eval_metrics_all.csv
Wrote 672 rows to output/ablations_fixed_longer_sortfree/trex_checkpoint_eval_metrics_all.csv
Wrote 352 rows to output/ablations_no_usplat_dropout_off/trex_checkpoint_eval_metrics_all.csv
Wrote 352 rows to output/ablations_no_usplat_dropout_on/trex_checkpoint_eval_metrics_all.csv
Skipping, no metrics file: output/ablations_usplat_sort_dropout_off/trex/anisotropic--rgb--sort--no_pruning--ess/checkpoint_eval_metrics.cs

In [3]:
def prt(df):
    print(df.to_string())

def filter_df(df: pd.DataFrame, **filters) -> pd.DataFrame:
    out = df

    for col, value in filters.items():
        if col not in df:
            raise KeyError(f"Missing column: {col}")

        values = value if isinstance(value, (list, tuple, set)) else [value]
        valid = sorted(out[col].dropna().unique())

        out = out[out[col].isin(values)]

        if out.empty:
            print(
                f"{col} looking for {values}. \n"
                f"Valid options: {valid}"
            )
    prt(out)
    return out.copy()

In [ ]:
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

def smooth_line_numpy(xs, ys, points=200, window=3):
    """
    Lightweight smoothing without scipy.
    Uses linear interpolation + moving average without endpoint drop.
    """
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)

    valid = np.isfinite(xs) & np.isfinite(ys)
    xs = xs[valid]
    ys = ys[valid]

    if len(xs) < 3:
        return xs, ys

    order = np.argsort(xs)
    xs = xs[order]
    ys = ys[order]

    x_smooth = np.linspace(xs.min(), xs.max(), points)
    y_interp = np.interp(x_smooth, xs, ys)

    window = max(1, int(window))
    if window <= 1:
        return x_smooth, y_interp

    kernel = np.ones(window) / window

    pad_left = window // 2
    pad_right = window - 1 - pad_left

    y_padded = np.pad(
        y_interp,
        pad_width=(pad_left, pad_right),
        mode="edge",
    )

    y_smooth = np.convolve(y_padded, kernel, mode="valid")

    y_smooth[0] = y_interp[0]
    y_smooth[-1] = y_interp[-1]

    return x_smooth, y_smooth


def plot_training_1x3(
    df,
    x_col="eval_checkpoint_iteration",
    run_col="run_hash",
    plot_specs=None,
    dataset_name="",
):
    """
    plot_specs: list of dicts, one per subplot row. Each dict:
      metric      – str, column name for y values
      group_cols  – tuple of 2 str column names that form the group key
      color_map   – dict mapping (val1, val2) -> hex color
      ylabel      – str for y-axis label
      invert      – bool, invert y-axis (default False)
      y_scale     – float divisor applied to y values (default 1.0)
    """
    if plot_specs is None:
        plot_specs = []

    def norm(v):
        return str(v).strip().lower().replace("-", "_").replace(" ", "_")

    fig, axes = plt.subplots(len(plot_specs), 1, figsize=(8, 9), sharex=True)
    if len(plot_specs) == 1:
        axes = [axes]

    fig.subplots_adjust(left=0.09, right=0.68, bottom=0.07, top=0.93, hspace=0.15)

    for ax, spec in zip(axes, plot_specs):
        metric     = spec["metric"]
        group_cols = spec["group_cols"]
        color_map  = spec["color_map"]
        ylabel     = spec.get("ylabel", metric.upper())
        invert     = spec.get("invert", False)
        y_scale    = spec.get("y_scale", 1.0)

        if metric not in df.columns or any(c not in df.columns for c in group_cols):
            ax.set_visible(False)
            continue

        skipped = set()

        for _, run_df in df.groupby(run_col):
            run_df = run_df.sort_values(x_col)
            first_valid = run_df.dropna(subset=list(group_cols))
            if first_valid.empty:
                continue

            raw_key  = tuple(first_valid[c].iloc[0] for c in group_cols)
            norm_key = tuple(norm(v) for v in raw_key)

            color = next(
                (c for k, c in color_map.items() if tuple(norm(v) for v in k) == norm_key),
                None,
            )
            if color is None:
                skipped.add(norm_key)
                continue

            xs = run_df[x_col].to_numpy(float) / 1000.0
            ys = run_df[metric].to_numpy(float) / y_scale
            mask = np.isfinite(xs) & np.isfinite(ys)
            xs, ys = xs[mask], ys[mask]
            if len(xs) == 0:
                continue

            x_s, y_s = smooth_line_numpy(xs, ys)
            ax.plot(x_s, y_s, color=color, lw=2)
            ax.scatter(xs, ys, s=18, color=color)

        if skipped:
            print(f"Skipped unrecognized groups for {metric}: {skipped}")

        if invert:
            ax.invert_yaxis()

        ax.set_ylabel(ylabel, fontsize=20, fontweight="bold")
        ax.tick_params(labelsize=12)
        ax.grid(alpha=0.3)
        ax.margins(x=0, y=0.03)

        ax.legend(
            handles=[
                Line2D([0], [0], color=c, lw=2, label=" / ".join(str(v) for v in k))
                for k, c in color_map.items()
            ],
            loc="center left",
            bbox_to_anchor=(1.01, 0.5),
            fontsize=12,
            frameon=True,
            borderaxespad=0.0,
        )

    axes[-1].set_xlabel(r"Checkpoint iteration, $n \times 10^3$", fontsize=16)
    plot_center_x = (fig.subplotpars.left + fig.subplotpars.right) / 2
    fig.suptitle(dataset_name, fontsize=24, fontweight="bold", x=plot_center_x, y=0.995)
    plt.show()
    


# Metrics

In [ ]:
df_clean = df_clean[df_clean["dataset_key"] == "ablations_fixed_longer/trex"]

In [ ]:
df_clean[df_clean["dataset_key"] == "ablations_fixed_longer/trex"]["sorting"].value_counts()

## Visual

In [ ]:
plot_training_1x3(
    filter_df(
        df_clean,
        scene_name="trex",
        use_usplat=False,
    ),
    dataset_name="trex",
    plot_specs=[
        {
            "metric":     "psnr",
            "group_cols": ("sorting", "dropout"),
            "color_map": {
                ("sort",      "no_dropout"):  "#0b3c8c",
                ("sort",      "yes_dropout"): "#6fa8dc",
                ("sort_free", "no_dropout"):  "#4f0f0f",
                ("sort_free", "yes_dropout"): "#e06666",
            },
            "ylabel": "PSNR",
        },
        {
            "metric":     "lpips",
            "group_cols": ("sorting", "dropout"),
            "color_map": {
                ("sort",      "no_dropout"):  "#0b3c8c",
                ("sort",      "yes_dropout"): "#6fa8dc",
                ("sort_free", "no_dropout"):  "#4f0f0f",
                ("sort_free", "yes_dropout"): "#e06666",
            },
            "ylabel": "LPIPS",
            "invert": False,
        },
        {
            "metric":     "ssim",
            "group_cols": ("sorting", "dropout"),
            "color_map": {
                ("sort",      "no_dropout"):  "#0b3c8c",
                ("sort",      "yes_dropout"): "#6fa8dc",
                ("sort_free", "no_dropout"):  "#4f0f0f",
                ("sort_free", "yes_dropout"): "#e06666",
            },
            "ylabel": "SSIM",
        },
    ],
)

In [ ]:
# lower band 

filtered = filter_df(
    df_clean,
    scene_name="trex",
    use_usplat=False,
)

prt(
    filtered[
        (filtered["checkpoint_name_iteration"] == 4000) &
        (filtered["lpips"] > 0.3)
    ].sort_values("lpips", ascending=False)[ABLATION_COLS]
)

## Memory

In [ ]:
# no_pruning
plot_training_1x3(
    filter_df(df_clean, scene_name="trex", pruning="no_pruning"),
    dataset_name="trex — no_pruning",
    plot_specs=[
        {
            "metric":     "peak_eval_vram_mb",
            "group_cols": ("appearance", "pruning"),
            "color_map": {
                ("rgb", "no_pruning"): "#00c853",
                ("sh3", "no_pruning"): "#b00020",
            },
            "ylabel": "VRAM",
            "invert": True,
        },
        {
            "metric":     "checkpoint_size_mb",
            "group_cols": ("sorting", "appearance"),
            "color_map": {
                ("sort_free", "rgb"): "#ff1744",
                ("sort_free", "sh3"): "#b00020",
                ("sort",      "rgb"): "#00c853",
                ("sort",      "sh3"): "#234aaa",
            },
            "ylabel": "SIZE",
            "invert": True,
        },
        {
            "metric":     "final_gaussian_count",
            "group_cols": ("pruning", "appearance"),
            "color_map": {
                ("no_pruning", "rgb"): "#00c853",
                ("no_pruning", "sh3"): "#234aaa",
            },
            "ylabel": r"Gaussians $\times\,10^4$",
            "invert":  True,
            "y_scale": 10_000.0,
        },
    ],
)

# interleaved_prune_densify
plot_training_1x3(
    filter_df(df_clean, scene_name="trex", pruning="interleaved_prune_densify"),
    dataset_name="trex — interleaved_prune_densify",
    plot_specs=[
        {
            "metric":     "peak_eval_vram_mb",
            "group_cols": ("appearance", "pruning"),
            "color_map": {
                ("rgb", "interleaved"): "#0047ff",
                ("sh3", "interleaved"): "#ff1744",
            },
            "ylabel": "VRAM",
            "invert": True,
        },
        {
            "metric":     "checkpoint_size_mb",
            "group_cols": ("sorting", "appearance"),
            "color_map": {
                ("sort_free", "rgb"): "#ff1744",
                ("sort_free", "sh3"): "#b00020",
                ("sort",      "rgb"): "#00c853",
                ("sort",      "sh3"): "#234aaa",
            },
            "ylabel": "SIZE",
            "invert": True,
        },
        {
            "metric":     "final_gaussian_count",
            "group_cols": ("pruning", "appearance"),
            "color_map": {
                ("interleaved_prune_densify", "rgb"): "#ff1744",
                ("interleaved_prune_densify", "sh3"): "#b00020",
            },
            "ylabel": r"Gaussians $\times\,10^4$",
            "invert":  True,
            "y_scale": 10_000.0,
        },
    ],
)

In [ ]:
filter_df(df_clean, scene_name="trex", pruning="interleaved_prune_densify")["peak_eval_vram_mb"]

In [ ]:
df_clean["pruning"].unique()

## Framerate

df_clean["]

# Ablations

## Sorting

## Appearance

## Pruning

## ESS

## Dropout

In [31]:
import pandas as pd


def make_typst_ablation_table(
    ablations_df: pd.DataFrame,
    metrics_df: pd.DataFrame,
    psnr_bad_below: float = -1e100,
    ssim_bad_below: float = -1e100,
    lpips_bad_above: float = 1e100,
) -> str:
    def latest_per_run(df: pd.DataFrame) -> pd.DataFrame:
        return (
            df.sort_values("eval_checkpoint_iteration")
            .groupby("run_hash", as_index=False)
            .tail(1)
            .reset_index(drop=True)
        )

    def esc(x):
        if pd.isna(x):
            return ""
        return (
            str(x)
            .replace("\\", "\\\\")
            .replace("[", "\\[")
            .replace("]", "\\]")
            .replace("#", "\\#")
        )

    def short(x):
        mapping = {
            "anisotropic": "aniso",
            "isotropic": "iso",
            "rgb": "RGB",
            "sh3": "SH3",
            "sort": "sort",
            "no_pruning": "none",
            "interleaved_prune_densify": "prune+dense",
            "dropout": "yes",
            "no_dropout": "no",
            "ess": "yes",
            "no_ess": "no",
            True: "yes",
            False: "no",
        }
        return mapping.get(x, x)

    def fmt(col, x):
        if pd.isna(x):
            return ""

        if col == "psnr":
            return f"{x:.2f}"
        if col in ["ssim", "lpips"]:
            return f"{x:.3f}"
        if col == "render_fps":
            return f"{x:.0f}"
        if col == "peak_eval_vram_mb":
            return f"{x:.0f}"
        if col == "checkpoint_size_mb":
            return f"{x:.1f}"
        if col == "final_gaussian_count":
            return f"{x / 1000:.1f}"
        if col == "training_wall_clock_sec_at_checkpoint":
            return f"{x / 60:.0f}m"
        if col == "eval_checkpoint_iteration":
            return f"{x / 1000:.0f}"

        return str(short(x))

    def is_bad_metric(col, x) -> bool:
        if pd.isna(x):
            return False

        x = float(x)

        if col == "psnr":
            return x < psnr_bad_below

        if col == "ssim":
            return x < ssim_bad_below

        if col == "lpips":
            return x > lpips_bad_above

        return False

    def ablation_color(col, x):
        v = str(short(x)).lower()

        if v in ["yes", "true"]:
            return "#d9efdf"  # green

        if v in ["no", "false", "none"]:
            return "#f2f2f2"  # grey

        if v in ["iso", "isotropic"]:
            return "#e3f2fd"  # blue-ish

        if v in ["aniso", "anisotropic"]:
            return "#fff4bf"  # yellow

        if v in ["rgb"]:
            return "#fce4ec"  # pink-ish

        if v in ["sh3"]:
            return "#ede7f6"  # purple-ish

        if v in ["prune+dense"]:
            return "#ffe0b2"  # orange

        return "#ffffff"

    ablations_df = latest_per_run(ablations_df)
    metrics_df = latest_per_run(metrics_df)

    df = ablations_df.merge(
        metrics_df,
        on="run_hash",
        how="left",
        suffixes=("", "_metric"),
    )

    ablation_candidates = [
        "isotropy",
        "sorting",
        "appearance",
        "pruning",
        "dropout",
        "ess",
        "use_usplat",
    ]

    ablation_cols = []
    for c in ablation_candidates:
        if c in df.columns and df[c].nunique(dropna=False) > 1:
            ablation_cols.append(c)

    if ablation_cols:
        df = df.sort_values(ablation_cols).reset_index(drop=True)

    metric_cols = [
        c for c in [
            "eval_checkpoint_iteration",
            "psnr",
            "ssim",
            "lpips",
            "render_fps",
            "peak_eval_vram_mb",
            "checkpoint_size_mb",
            "final_gaussian_count",
            "training_wall_clock_sec_at_checkpoint",
        ]
        if c in df.columns
    ]

    best_metric_cols = {
        "psnr": "max",
        "ssim": "max",
        "lpips": "min",
        "render_fps": "max",
        "peak_eval_vram_mb": "min",
        "checkpoint_size_mb": "min",
    }

    best_values = {}
    for c, direction in best_metric_cols.items():
        if c in df.columns:
            s = pd.to_numeric(df[c], errors="coerce")
            best_values[c] = s.max() if direction == "max" else s.min()

    names = {
        "isotropy": "Isotropy",
        "sorting": "Sorting",
        "appearance": "Color",
        "pruning": "Prune",
        "dropout": "Drop",
        "ess": "ESS",
        "use_usplat": "USplat",
        "eval_checkpoint_iteration": "Iter  ×10³",
        "psnr": "PSNR ↑",
        "ssim": "SSIM ↑",
        "lpips": "LPIPS ↓",
        "render_fps": "FPS ↑",
        "peak_eval_vram_mb": "VRAM ↓",
        "checkpoint_size_mb": "MB ↓",
        "final_gaussian_count": "Gauss k ↓",
        "training_wall_clock_sec_at_checkpoint": "Train m ↓",
    }

    scene = df["scene_name"].iloc[0] if "scene_name" in df.columns else "Ablations"

    columns = ablation_cols + metric_cols

    n_ablation = len(ablation_cols)
    n_visual = len(metric_cols)

    col_widths = ["0.7fr"] * n_ablation + ["0.65fr"] * n_visual

    lines = []

    lines.append("#figure(")
    lines.append("  block(width: 100%)[")
    lines.append("    #set text(size: 6.5pt)")
    lines.append("")
    lines.append("    #let dark = rgb(\"#2c3e50\")")
    lines.append("    #let section = rgb(\"#f2f2f2\")")
    lines.append("    #let border = rgb(\"#cccccc\")")
    lines.append("    #let bad = rgb(\"#ffd6d6\")")
    lines.append("    #let bad-text = rgb(\"#b00020\")")
    lines.append("")
    lines.append("    #table(")
    lines.append(f"      columns: ({', '.join(col_widths)}),")
    lines.append("      stroke: border,")
    lines.append("      inset: 3pt,")
    lines.append("      align: center + horizon,")
    lines.append("")

    if n_ablation > 0:
        lines.append(
            f"      table.cell(colspan: {n_ablation}, fill: section)"
            "[#text(weight: \"bold\")[ABLATIONS]],"
        )

    lines.append(
        f"      table.cell(colspan: {n_visual}, fill: section)"
        "[#text(weight: \"bold\")[VISUAL]],"
    )
    lines.append("")

    for c in columns:
        lines.append(
            f"      table.cell(fill: dark)"
            f"[#text(fill: white, weight: \"bold\")[{esc(names.get(c, c))}]],"
        )

    lines.append("")

    for _, row in df.iterrows():
        cells = []

        for c in columns:
            value = esc(fmt(c, row[c]))

            if c in ablation_cols:
                color = ablation_color(c, row[c])
                cells.append(f'table.cell(fill: rgb("{color}"))[{value}]')

            elif is_bad_metric(c, row[c]):
                cells.append(
                    f'table.cell(fill: bad)[#text(fill: bad-text, weight: "bold")[{value}]]'
                )

            elif c in best_values and pd.notna(row[c]) and row[c] == best_values[c]:
                cells.append(f'table.cell(fill: rgb("#d9efdf"))[{value}]')

            else:
                cells.append(f"[{value}]")

        lines.append("      " + ", ".join(cells) + ",")

    lines.append("    )")
    lines.append("  ],")
    lines.append(f"  caption: [{esc(scene)} ablation results.],")
    lines.append(")")

    return "\n".join(lines)

In [34]:
from pathlib import Path

ablation_columns = [
    "run_hash",
    "scene_name",
    "variant_name",
    "matrix_preset",
    "isotropy",
    "appearance",
    "sorting",
    "pruning",
    "dropout",
    "ess",
    "use_usplat",
    "eval_checkpoint_iteration",
]

metric_columns = [
    "run_hash",
    "eval_checkpoint_iteration",
    "psnr",
    "ssim",
    "lpips",
    "render_fps",
    "peak_eval_vram_mb",
    "final_gaussian_count",
    "checkpoint_size_mb",
    "training_wall_clock_sec_at_checkpoint",
]

out_files = []


twentyk = (df_clean["dataset_key"] == "ablations_fixed_longer/trex") | (df_clean["dataset_key"] == "ablations_fixed_longer_sortfree/trex")
df = df_clean[twentyk]

ablations_df = df[ablation_columns].copy()
metrics_df = df[metric_columns].copy()

typst_table = make_typst_ablation_table(
    ablations_df,
    metrics_df,
    psnr_bad_below=20,
    ssim_bad_below=0.9,
    lpips_bad_above=0.1,
)

print(typst_table)

#figure(
  block(width: 100%)[
    #set text(size: 6.5pt)

    #let dark = rgb("#2c3e50")
    #let section = rgb("#f2f2f2")
    #let border = rgb("#cccccc")
    #let bad = rgb("#ffd6d6")
    #let bad-text = rgb("#b00020")

    #table(
      columns: (0.7fr, 0.7fr, 0.7fr, 0.7fr, 0.7fr, 0.7fr, 0.65fr, 0.65fr, 0.65fr, 0.65fr, 0.65fr, 0.65fr, 0.65fr, 0.65fr, 0.65fr),
      stroke: border,
      inset: 3pt,
      align: center + horizon,

      table.cell(colspan: 6, fill: section)[#text(weight: "bold")[ABLATIONS]],
      table.cell(colspan: 9, fill: section)[#text(weight: "bold")[VISUAL]],

      table.cell(fill: dark)[#text(fill: white, weight: "bold")[Isotropy]],
      table.cell(fill: dark)[#text(fill: white, weight: "bold")[Sorting]],
      table.cell(fill: dark)[#text(fill: white, weight: "bold")[Color]],
      table.cell(fill: dark)[#text(fill: white, weight: "bold")[Prune]],
      table.cell(fill: dark)[#text(fill: white, weight: "bold")[Drop]],
      table.cell(fill: dark)[#text(fi

In [38]:
prt(df.head())

              run_hash                  dataset_key scene_name                                                     variant_name variant_index matrix_preset     isotropy use_usplat appearance sorting                    pruning  dropout  ess checkpoint_index checkpoint_name_iteration eval_checkpoint_iteration       psnr      ssim     lpips   render_fps peak_eval_vram_mb checkpoint_size_mb training_wall_clock_sec_at_checkpoint final_gaussian_count
1056  af7b4fb353a95d3a  ablations_fixed_longer/trex       trex  anisotropic__rgb__sort__interleaved_prune_densify__dropout__ess             0     cartesian  anisotropic      False        rgb    sort  interleaved_prune_densify  dropout  ess                0                    1000.0                      1000  24.896741  0.941057  0.136209  2329.090739         54.047852           1.804128                             17.832233                 7004
1057  af7b4fb353a95d3a  ablations_fixed_longer/trex       trex  anisotropic__rgb__sort__interleaved_pr

In [60]:
import numpy as np
import pandas as pd


LOWER_IS_BETTER = {
    "lpips",
    "peak_eval_vram_mb",
    "final_gaussian_count",
    "checkpoint_size_mb",
    "training_wall_clock_sec_at_checkpoint",
}

METRIC_LABELS = {
    "psnr": "PSNR",
    "ssim": "SSIM",
    "lpips": "LPIPS",
    "render_fps": "render FPS",
    "peak_eval_vram_mb": "eval VRAM",
    "final_gaussian_count": "Gaussian count",
    "checkpoint_size_mb": "checkpoint size",
    "training_wall_clock_sec_at_checkpoint": "training time",
}

METRIC_EXCLUDE = {
    "run_hash",
    "checkpoint_index",
    "checkpoint_name_iteration",
    "eval_checkpoint_iteration",
    "variant_index",
}


def cohens_d(x: pd.Series, y: pd.Series) -> float:
    pooled_var = (
        ((len(x) - 1) * x.var(ddof=1) + (len(y) - 1) * y.var(ddof=1))
        / (len(x) + len(y) - 2)
    )
    pooled_std = np.sqrt(pooled_var)
    return (x.mean() - y.mean()) / pooled_std if pooled_std > 0 else np.nan


def robust_d(x: pd.Series, y: pd.Series) -> float:
    x_med = x.median()
    y_med = y.median()
    pooled_mad = np.nanmean([
        (x - x_med).abs().median(),
        (y - y_med).abs().median(),
    ])
    return (x_med - y_med) / (1.4826 * pooled_mad) if pooled_mad > 0 else np.nan


def analyze_ablation_effects(
    df: pd.DataFrame,
    ablation_columns: list[str],
    metric_columns: list[str],
    *,
    ablation_group_columns: list[str] | None = None,
    min_abs_effect: float = 0.2,
    return_all: bool = False,
) -> pd.DataFrame:
    df = df.copy()

    metrics = [
        c for c in metric_columns
        if c in df.columns and c not in METRIC_EXCLUDE
    ]

    for c in metrics:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    metrics = [c for c in metrics if df[c].notna().any()]

    if ablation_group_columns is None:
        ablation_group_columns = [
            c for c in ablation_columns
            if c in df.columns
            and c not in {
                "run_hash",
                "scene_name",
                "variant_name",
                "eval_checkpoint_iteration",
            }
        ]

    rows = []

    for ablation_col in ablation_group_columns:
        for ablation_value, group in df.groupby(ablation_col, dropna=False):
            other = (
                df[df[ablation_col].notna()]
                if pd.isna(ablation_value)
                else df[df[ablation_col] != ablation_value]
            )

            if other.empty:
                continue

            for metric in metrics:
                x = group[metric].dropna()
                y = other[metric].dropna()

                if len(x) < 2 or len(y) < 2:
                    continue

                d = cohens_d(x, y)
                rd = robust_d(x, y)

                effects = [abs(v) for v in [d, rd] if pd.notna(v)]
                if not effects:
                    continue

                abs_effect = max(effects)
                if not return_all and abs_effect < min_abs_effect:
                    continue

                sign = -1 if metric in LOWER_IS_BETTER else 1

                raw_delta = x.mean() - y.mean()
                benefit_delta = sign * raw_delta
                benefit_d = sign * d if pd.notna(d) else np.nan
                benefit_rd = sign * rd if pd.notna(rd) else np.nan

                rows.append({
                    "ablation_column": ablation_col,
                    "ablation_value": ablation_value,
                    "metric": metric,
                    "n": len(x),
                    "other_n": len(y),

                    "group_mean": x.mean(),
                    "other_mean": y.mean(),
                    "delta_mean": raw_delta,
                    "pct_delta_mean": 100 * raw_delta / abs(y.mean()) if y.mean() else np.nan,

                    "group_median": x.median(),
                    "other_median": y.median(),
                    "delta_median": x.median() - y.median(),

                    "cohens_d": d,
                    "robust_effect": rd,
                    "benefit_delta_mean": benefit_delta,
                    "benefit_effect": benefit_d,
                    "benefit_robust_effect": benefit_rd,
                    "abs_effect": abs_effect,

                    "interpretation": (
                        "better" if benefit_delta > 0
                        else "worse" if benefit_delta < 0
                        else "same"
                    ),
                })

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    return out.sort_values(
        ["abs_effect", "ablation_column", "metric"],
        ascending=[False, True, True],
    ).reset_index(drop=True)


def fmt_value(metric: str, x: float) -> str:
    if pd.isna(x):
        return "nan"
    if metric in {"ssim", "lpips"}:
        return f"{x:.4f}"
    if metric == "psnr":
        return f"{x:.2f}"
    if metric == "render_fps":
        return f"{x:.1f}"
    if metric == "final_gaussian_count":
        return f"{x:,.0f}"
    return f"{x:.1f}"


def effect_strength(d: float) -> str:
    if d >= 2.0:
        return "very large"
    if d >= 1.0:
        return "large"
    if d >= 0.5:
        return "moderate"
    return "small"


def metric_list(g: pd.DataFrame, interpretation: str) -> str:
    rows = (
        g[g["interpretation"].eq(interpretation)]
        .sort_values("abs_d", ascending=False)
    )

    if rows.empty:
        return "none"

    parts = []

    for _, r in rows.iterrows():
        sign = "+" if r["benefit_d"] > 0 else ""
        parts.append(f"{r['metric_label']} ({sign}{r['benefit_d']:.2f})")

    return ", ".join(parts)


def ablation_effects_to_text(
    table: pd.DataFrame,
    *,
    top_n: int = 40,
    min_abs_d: float = 0.5,
    collapse_mirrors: bool = True,
    print_output: bool = True,
) -> str:
    if table.empty:
        text = "No notable ablation effects found."
        if print_output:
            print(text)
        return text

    df = table.copy()
    df["metric_label"] = df["metric"].map(METRIC_LABELS).fillna(df["metric"])
    df["abs_d"] = df["cohens_d"].abs()
    df["benefit_d"] = np.where(
        df["interpretation"].eq("better"),
        df["abs_d"],
        -df["abs_d"],
    )

    df = df[df["abs_d"] >= min_abs_d].copy()

    if df.empty:
        text = f"No ablation effects with |d| >= {min_abs_d}."
        if print_output:
            print(text)
        return text

    lines = ["=== Ablation summary ===", "", f"Significant ablation effects, |d| >= {min_abs_d}:"]

    specifics = (
        df.groupby(["ablation_column", "ablation_value"], dropna=False)
        .agg(
            net_d=("benefit_d", "mean"),
            mean_abs_d=("abs_d", "mean"),
        )
        .reset_index()
        .sort_values("net_d", ascending=False)
    )

    for _, r in specifics.iterrows():
        g = df[
            df["ablation_column"].eq(r["ablation_column"])
            & df["ablation_value"].eq(r["ablation_value"])
        ]

        net_d = r["net_d"]
        verdict = (
            "mostly beneficial" if net_d > 0.15
            else "mostly harmful" if net_d < -0.15
            else "mixed"
        )

        lines.append(
            f"- {r['ablation_column']}={r['ablation_value']}: "
            f"{verdict}; "
            f"better on {metric_list(g, 'better')}; "
            f"worse on {metric_list(g, 'worse')}; "
            f"net d={net_d:.2f}, mean |d|={r['mean_abs_d']:.2f}"
        )

    if collapse_mirrors:
        kept = []

        for _, g in df.groupby(["ablation_column", "metric"], dropna=False):
            better = g[g["interpretation"].eq("better")]
            pick_from = better if not better.empty else g
            kept.append(pick_from.sort_values("abs_d", ascending=False).iloc[0])

        df = pd.DataFrame(kept)

    df = df.sort_values("abs_d", ascending=False).head(top_n)

    lines += ["", "Strongest individual effects:"]

    for _, r in df.iterrows():
        metric = r["metric"]
        verb = "improves" if r["interpretation"] == "better" else "hurts"
        pct = "nan%" if pd.isna(r["pct_delta_mean"]) else f"{r['pct_delta_mean']:+.1f}%"
        direction = "lower is better" if metric in LOWER_IS_BETTER else "higher is better"

        lines.append(
            f"- {r['ablation_column']}={r['ablation_value']} {verb} "
            f"{r['metric_label']}: "
            f"{fmt_value(metric, r['group_mean'])} vs {fmt_value(metric, r['other_mean'])} "
            f"({pct}, {effect_strength(r['abs_d'])}, d={r['cohens_d']:.2f}; "
            f"{direction})"
        )

    text = "\n".join(lines)

    if print_output:
        print(text)

    return text


report = analyze_ablation_effects(
    df,
    ablation_columns=ablation_columns,
    metric_columns=metric_columns,
    min_abs_effect=0.2,
)

notable = report[report["abs_effect"] >= 0.5].copy()

text = ablation_effects_to_text(
    notable,
    top_n=40,
    min_abs_d=0.5,
    collapse_mirrors=True,
)

=== Ablation summary ===

Significant ablation effects, |d| >= 0.5:
- appearance=rgb: mostly beneficial; better on checkpoint size (+4.91), eval VRAM (+3.66), Gaussian count (+0.93), training time (+0.85), render FPS (+0.55); worse on PSNR (-1.50), LPIPS (-1.38), SSIM (-0.93); net d=0.89, mean |d|=1.84
- sorting=sort: mostly beneficial; better on LPIPS (+1.46), render FPS (+1.34), PSNR (+1.33), SSIM (+1.30), training time (+1.01), eval VRAM (+0.65); worse on Gaussian count (-1.20); net d=0.84, mean |d|=1.19
- sorting=sort_free: mostly harmful; better on Gaussian count (+1.20); worse on LPIPS (-1.46), render FPS (-1.34), PSNR (-1.33), SSIM (-1.30), training time (-1.01), eval VRAM (-0.65); net d=-0.84, mean |d|=1.19
- appearance=sh3: mostly harmful; better on PSNR (+1.50), LPIPS (+1.38), SSIM (+0.93); worse on checkpoint size (-4.91), eval VRAM (-3.66), Gaussian count (-0.93), training time (-0.85), render FPS (-0.55); net d=-0.89, mean |d|=1.84

Strongest individual effects:
- appearan

In [62]:
import numpy as np
import pandas as pd


LOWER_IS_BETTER = {
    "lpips",
    "peak_eval_vram_mb",
    "final_gaussian_count",
    "checkpoint_size_mb",
    "training_wall_clock_sec_at_checkpoint",
}

METRIC_LABELS = {
    "psnr": "PSNR",
    "ssim": "SSIM",
    "lpips": "LPIPS",
    "render_fps": "render FPS",
    "peak_eval_vram_mb": "eval VRAM",
    "final_gaussian_count": "Gaussian count",
    "checkpoint_size_mb": "checkpoint size",
    "training_wall_clock_sec_at_checkpoint": "training time",
}

METRIC_EXCLUDE = {
    "run_hash",
    "checkpoint_index",
    "checkpoint_name_iteration",
    "eval_checkpoint_iteration",
    "variant_index",
}


def cohens_d(x: pd.Series, y: pd.Series) -> float:
    n_x = len(x)
    n_y = len(y)

    if n_x < 2 or n_y < 2:
        return np.nan

    pooled_var = (
        ((n_x - 1) * x.var(ddof=1) + (n_y - 1) * y.var(ddof=1))
        / (n_x + n_y - 2)
    )

    pooled_std = np.sqrt(pooled_var)
    return (x.mean() - y.mean()) / pooled_std if pooled_std > 0 else np.nan


def hedges_g(x: pd.Series, y: pd.Series) -> float:
    """
    Hedges' g = Cohen's d with small-sample bias correction.
    Independent-groups version.
    """
    n_x = len(x)
    n_y = len(y)
    df = n_x + n_y - 2

    if n_x < 2 or n_y < 2 or df <= 1:
        return np.nan

    d = cohens_d(x, y)

    if pd.isna(d):
        return np.nan

    correction = 1 - (3 / (4 * df - 1))
    return correction * d


def robust_g(x: pd.Series, y: pd.Series) -> float:
    """
    Robust standardized median difference.

    This is not technically Hedges' g, but is kept as a robust companion
    to Hedges' g, analogous to your previous robust_d.
    """
    x_med = x.median()
    y_med = y.median()

    pooled_mad = np.nanmean([
        (x - x_med).abs().median(),
        (y - y_med).abs().median(),
    ])

    return (x_med - y_med) / (1.4826 * pooled_mad) if pooled_mad > 0 else np.nan


def analyze_ablation_effects_hedges_g(
    df: pd.DataFrame,
    ablation_columns: list[str],
    metric_columns: list[str],
    *,
    ablation_group_columns: list[str] | None = None,
    min_abs_effect: float = 0.2,
    return_all: bool = False,
) -> pd.DataFrame:
    df = df.copy()

    metrics = [
        c for c in metric_columns
        if c in df.columns and c not in METRIC_EXCLUDE
    ]

    for c in metrics:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    metrics = [c for c in metrics if df[c].notna().any()]

    if ablation_group_columns is None:
        ablation_group_columns = [
            c for c in ablation_columns
            if c in df.columns
            and c not in {
                "run_hash",
                "scene_name",
                "variant_name",
                "eval_checkpoint_iteration",
            }
        ]

    rows = []

    for ablation_col in ablation_group_columns:
        for ablation_value, group in df.groupby(ablation_col, dropna=False):
            other = (
                df[df[ablation_col].notna()]
                if pd.isna(ablation_value)
                else df[df[ablation_col] != ablation_value]
            )

            if other.empty:
                continue

            for metric in metrics:
                x = group[metric].dropna()
                y = other[metric].dropna()

                if len(x) < 2 or len(y) < 2:
                    continue

                g = hedges_g(x, y)
                rg = robust_g(x, y)

                effects = [abs(v) for v in [g, rg] if pd.notna(v)]
                if not effects:
                    continue

                abs_effect = max(effects)

                if not return_all and abs_effect < min_abs_effect:
                    continue

                sign = -1 if metric in LOWER_IS_BETTER else 1

                raw_delta = x.mean() - y.mean()
                benefit_delta = sign * raw_delta
                benefit_g = sign * g if pd.notna(g) else np.nan
                benefit_rg = sign * rg if pd.notna(rg) else np.nan

                rows.append({
                    "ablation_column": ablation_col,
                    "ablation_value": ablation_value,
                    "metric": metric,
                    "n": len(x),
                    "other_n": len(y),

                    "group_mean": x.mean(),
                    "other_mean": y.mean(),
                    "delta_mean": raw_delta,
                    "pct_delta_mean": 100 * raw_delta / abs(y.mean()) if y.mean() else np.nan,

                    "group_median": x.median(),
                    "other_median": y.median(),
                    "delta_median": x.median() - y.median(),

                    "hedges_g": g,
                    "robust_effect": rg,
                    "benefit_delta_mean": benefit_delta,
                    "benefit_effect": benefit_g,
                    "benefit_robust_effect": benefit_rg,
                    "abs_effect": abs_effect,

                    "interpretation": (
                        "better" if benefit_delta > 0
                        else "worse" if benefit_delta < 0
                        else "same"
                    ),
                })

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    return out.sort_values(
        ["abs_effect", "ablation_column", "metric"],
        ascending=[False, True, True],
    ).reset_index(drop=True)


def fmt_value(metric: str, x: float) -> str:
    if pd.isna(x):
        return "nan"
    if metric in {"ssim", "lpips"}:
        return f"{x:.4f}"
    if metric == "psnr":
        return f"{x:.2f}"
    if metric == "render_fps":
        return f"{x:.1f}"
    if metric == "final_gaussian_count":
        return f"{x:,.0f}"
    return f"{x:.1f}"


def effect_strength(g: float) -> str:
    if g >= 2.0:
        return "very large"
    if g >= 1.0:
        return "large"
    if g >= 0.5:
        return "moderate"
    return "small"


def metric_list(gdf: pd.DataFrame, interpretation: str) -> str:
    rows = (
        gdf[gdf["interpretation"].eq(interpretation)]
        .sort_values("abs_g", ascending=False)
    )

    if rows.empty:
        return "none"

    parts = []

    for _, r in rows.iterrows():
        sign = "+" if r["benefit_g"] > 0 else ""
        parts.append(f"{r['metric_label']} ({sign}{r['benefit_g']:.2f})")

    return ", ".join(parts)


def ablation_effects_to_text_hedges_g(
    table: pd.DataFrame,
    *,
    top_n: int = 40,
    min_abs_g: float = 0.5,
    collapse_mirrors: bool = True,
    print_output: bool = True,
) -> str:
    if table.empty:
        text = "No notable ablation effects found."
        if print_output:
            print(text)
        return text

    df = table.copy()
    df["metric_label"] = df["metric"].map(METRIC_LABELS).fillna(df["metric"])
    df["abs_g"] = df["hedges_g"].abs()
    df["benefit_g"] = np.where(
        df["interpretation"].eq("better"),
        df["abs_g"],
        -df["abs_g"],
    )

    df = df[df["abs_g"] >= min_abs_g].copy()

    if df.empty:
        text = f"No ablation effects with |g| >= {min_abs_g}."
        if print_output:
            print(text)
        return text

    lines = [
        "=== Ablation summary ===",
        "",
        f"Significant ablation effects, |g| >= {min_abs_g}:",
    ]

    specifics = (
        df.groupby(["ablation_column", "ablation_value"], dropna=False)
        .agg(
            net_g=("benefit_g", "mean"),
            mean_abs_g=("abs_g", "mean"),
        )
        .reset_index()
        .sort_values("net_g", ascending=False)
    )

    for _, r in specifics.iterrows():
        gdf = df[
            df["ablation_column"].eq(r["ablation_column"])
            & df["ablation_value"].eq(r["ablation_value"])
        ]

        net_g = r["net_g"]
        verdict = (
            "mostly beneficial" if net_g > 0.15
            else "mostly harmful" if net_g < -0.15
            else "mixed"
        )

        lines.append(
            f"- {r['ablation_column']}={r['ablation_value']}: "
            f"{verdict}; "
            f"better on {metric_list(gdf, 'better')}; "
            f"worse on {metric_list(gdf, 'worse')}; "
            f"net g={net_g:.2f}, mean |g|={r['mean_abs_g']:.2f}"
        )

    if collapse_mirrors:
        kept = []

        for _, gdf in df.groupby(["ablation_column", "metric"], dropna=False):
            better = gdf[gdf["interpretation"].eq("better")]
            pick_from = better if not better.empty else gdf
            kept.append(pick_from.sort_values("abs_g", ascending=False).iloc[0])

        df = pd.DataFrame(kept)

    df = df.sort_values("abs_g", ascending=False).head(top_n)

    lines += ["", "Strongest individual effects:"]

    for _, r in df.iterrows():
        metric = r["metric"]
        verb = "improves" if r["interpretation"] == "better" else "hurts"
        pct = "nan%" if pd.isna(r["pct_delta_mean"]) else f"{r['pct_delta_mean']:+.1f}%"
        direction = "lower is better" if metric in LOWER_IS_BETTER else "higher is better"

        lines.append(
            f"- {r['ablation_column']}={r['ablation_value']} {verb} "
            f"{r['metric_label']}: "
            f"{fmt_value(metric, r['group_mean'])} vs {fmt_value(metric, r['other_mean'])} "
            f"({pct}, {effect_strength(r['abs_g'])}, g={r['hedges_g']:.2f}; "
            f"{direction})"
        )

    text = "\n".join(lines)

    if print_output:
        print(text)

    return text


report = analyze_ablation_effects_hedges_g(
    df,
    ablation_columns=ablation_columns,
    metric_columns=metric_columns,
    min_abs_effect=0.2,
)

notable = report[report["abs_effect"] >= 0.5].copy()

text = ablation_effects_to_text_hedges_g(
    notable,
    top_n=40,
    min_abs_g=0.5,
    collapse_mirrors=True,
)

=== Ablation summary ===

Significant ablation effects, |g| >= 0.5:
- appearance=rgb: mostly beneficial; better on checkpoint size (+4.91), eval VRAM (+3.66), Gaussian count (+0.93), training time (+0.85), render FPS (+0.55); worse on PSNR (-1.50), LPIPS (-1.37), SSIM (-0.93); net g=0.89, mean |g|=1.84
- sorting=sort: mostly beneficial; better on LPIPS (+1.46), render FPS (+1.34), PSNR (+1.33), SSIM (+1.30), training time (+1.01), eval VRAM (+0.65); worse on Gaussian count (-1.20); net g=0.84, mean |g|=1.19
- sorting=sort_free: mostly harmful; better on Gaussian count (+1.20); worse on LPIPS (-1.46), render FPS (-1.34), PSNR (-1.33), SSIM (-1.30), training time (-1.01), eval VRAM (-0.65); net g=-0.84, mean |g|=1.19
- appearance=sh3: mostly harmful; better on PSNR (+1.50), LPIPS (+1.37), SSIM (+0.93); worse on checkpoint size (-4.91), eval VRAM (-3.66), Gaussian count (-0.93), training time (-0.85), render FPS (-0.55); net g=-0.89, mean |g|=1.84

Strongest individual effects:
- appearan